# 02 – Data Preparation untuk Clustering

**Project:** Analisis Pola Persebaran Catatan Keanekaragaman Hayati di Indonesia Menggunakan Metode Clustering  
**Tahap ML Lifecycle:** [L] Learning Problem  
**Model yang disiapkan:** K-Means (baseline), Gaussian Mixture Model/GMM (advanced), dan Deep Embedded Clustering/DEC (deep learning).

**Perubahan penting dari notebook sebelumnya:** project ini menggunakan **unsupervised learning**. Karena itu, `class` tidak digunakan sebagai target prediksi, tidak dilakukan stratified split, dan tidak dilakukan SMOTE/undersampling kelas. Data mentah diagregasi menjadi satu baris per grid geografis dan tahun.

## 1. Import Library dan Menetapkan Konfigurasi

Bagian ini mengimpor library untuk pengolahan data, pembagian data berdasarkan kelompok wilayah, imputasi, scaling, serta penyimpanan hasil preprocessing. Ukuran grid ditetapkan 0,5 derajat dan periode kajian 2015–2025 sesuai rancangan project.

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import json
import joblib
from pathlib import Path
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
GRID_SIZE = 0.5
TAHUN_AWAL = 2015
TAHUN_AKHIR = 2025
OUTPUT_DIR = Path("/content/hasil_data_preparation_clustering")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
print("Library dan konfigurasi berhasil disiapkan.")
print("Ukuran grid:", GRID_SIZE, "derajat")
print("Periode:", TAHUN_AWAL, "-", TAHUN_AKHIR)

Library dan konfigurasi berhasil disiapkan.
Ukuran grid: 0.5 derajat
Periode: 2015 - 2025


## 2. Membaca Dataset GBIF

Notebook kedua membaca arsip GBIF yang sama seperti notebook pertama. Hanya atribut yang diperlukan untuk menghitung jumlah catatan dan karakteristik taksonomi yang dimuat agar proses lebih hemat memori.

In [ ]:
NAMA_ZIP = "0001923-261110092616502.zip"
FOLDER_DATA = Path("/content")
lokasi_zip = FOLDER_DATA / NAMA_ZIP

if not lokasi_zip.exists():
    kandidat = sorted(FOLDER_DATA.glob("*.zip"))
    kandidat_cocok = [p for p in kandidat if "0001923-261110092616502" in p.name]
    if kandidat_cocok:
        lokasi_zip = kandidat_cocok[0]
    elif len(kandidat) == 1:
        lokasi_zip = kandidat[0]
    else:
        raise FileNotFoundError(
            "ZIP GBIF tidak ditemukan. Upload file ZIP ke panel Files Google Colab "
            "atau sesuaikan variabel NAMA_ZIP."
        )

if not zipfile.is_zipfile(lokasi_zip):
    raise ValueError("File yang dipilih bukan arsip ZIP yang valid.")

kolom_model = [
    "gbifID", "kingdom", "class", "family", "species", "taxonRank",
    "countryCode", "occurrenceStatus", "decimalLatitude", "decimalLongitude",
    "year", "basisOfRecord", "coordinateUncertaintyInMeters", "depth"
]

with zipfile.ZipFile(lokasi_zip, "r") as arsip:
    daftar_data = [
        info for info in arsip.infolist()
        if info.filename.lower().endswith((".csv", ".tsv", ".txt")) and not info.is_dir()
    ]
    if not daftar_data:
        raise ValueError("Tidak ditemukan file tabel CSV/TSV/TXT di dalam ZIP.")
    info_data = max(daftar_data, key=lambda item: item.file_size)
    with arsip.open(info_data.filename) as f_header:
        kolom_sumber = pd.read_csv(f_header, sep="\t", nrows=0).columns.tolist()
    kolom_dibaca = [c for c in kolom_model if c in kolom_sumber]
    with arsip.open(info_data.filename) as f_data:
        df_raw = pd.read_csv(
            f_data, sep="\t", usecols=kolom_dibaca, low_memory=False
        )

kolom_wajib = ["gbifID", "kingdom", "species", "decimalLatitude", "decimalLongitude", "year"]
kolom_tidak_ada = [c for c in kolom_wajib if c not in df_raw.columns]
if kolom_tidak_ada:
    raise ValueError(f"Kolom wajib tidak ditemukan pada dataset: {kolom_tidak_ada}")

print("File:", info_data.filename)
print("Jumlah baris:", f"{len(df_raw):,}")
print("Kolom yang dibaca:", len(df_raw.columns))
display(df_raw.head())

File: 0001923-261110092616502.csv
Jumlah baris: 2,022,875
Kolom yang dibaca: 14


,gbifID,kingdom,class,family,species,taxonRank,countryCode,occurrenceStatus,decimalLatitude,decimalLongitude,coordinateUncertaintyInMeters,depth,year,basisOfRecord
0,3427261301,Plantae,Magnoliopsida,Annonaceae,Disepalum rawagambut,SPECIES,ID,DETECTED,-1.770700,104.201890,NaN,NaN,2020,MATERIAL_CITATION
1,4740657999,Animalia,Aves,Alcedinidae,Todiramphus chloris,SPECIES,ID,DETECTED,0.473349,121.437461,NaN,NaN,2023,HUMAN_OBSERVATION
2,4747492628,Animalia,Aves,Turnicidae,Turnix suscitator,SPECIES,ID,DETECTED,-0.414129,121.865931,NaN,NaN,2023,HUMAN_OBSERVATION
3,4749612951,Animalia,Aves,Rhipiduridae,Rhipidura leucophrys,SPECIES,ID,DETECTED,0.914723,127.547892,NaN,NaN,2023,HUMAN_OBSERVATION
4,4749648957,Animalia,Aves,Phasianidae,Arborophila javanica,SPECIES,ID,DETECTED,-7.158979,107.430856,NaN,NaN,2023,HUMAN_OBSERVATION


## 3. Data Cleaning – Pemeriksaan dan Seleksi Catatan

Data dibersihkan berdasarkan identitas catatan, negara, status kemunculan, tingkat taksonomi, kelengkapan nama spesies, koordinat, dan tahun. Tidak ada kolom target `y` karena setiap catatan akan digabungkan menjadi karakteristik grid-tahun untuk clustering.

In [ ]:
data = df_raw.copy()
jumlah_awal = len(data)
log_cleaning = []

# 1) Rapikan kolom teks yang digunakan.
for col in ["kingdom", "class", "family", "species", "taxonRank", "countryCode", "occurrenceStatus", "basisOfRecord"]:
    if col in data.columns:
        data[col] = data[col].astype("string").str.strip()

# 2) Hilangkan duplikasi berdasarkan identitas GBIF.
sebelum = len(data)
data = data.drop_duplicates(subset="gbifID", keep="first")
log_cleaning.append(["Duplikasi gbifID", sebelum - len(data), len(data)])

# 3) Batasi catatan pada Indonesia bila countryCode tersedia.
if "countryCode" in data.columns:
    sebelum = len(data)
    data = data[data["countryCode"].str.upper().eq("ID")].copy()
    log_cleaning.append(["Bukan countryCode ID", sebelum - len(data), len(data)])

# 4) Pilih catatan dengan status PRESENT bila kolom tersedia.
if "occurrenceStatus" in data.columns:
    nilai_status = set(data["occurrenceStatus"].dropna().str.upper().unique())
    if "PRESENT" in nilai_status:
        sebelum = len(data)
        data = data[data["occurrenceStatus"].str.upper().eq("PRESENT")].copy()
        log_cleaning.append(["Status bukan PRESENT", sebelum - len(data), len(data)])

# 5) Konversi kolom numerik.
for col in ["decimalLatitude", "decimalLongitude", "year", "coordinateUncertaintyInMeters", "depth"]:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors="coerce")

# 6) Batasi ke catatan tingkat spesies bila taxonRank tersedia.
if "taxonRank" in data.columns:
    sebelum = len(data)
    data = data[data["taxonRank"].str.upper().eq("SPECIES")].copy()
    log_cleaning.append(["Bukan taxonRank SPECIES", sebelum - len(data), len(data)])

# 7) Hapus catatan tanpa spesies, koordinat valid, atau tahun kajian.
sebelum = len(data)
data["species"] = data["species"].astype("string").str.strip()
data = data.dropna(subset=["species", "decimalLatitude", "decimalLongitude", "year"])
data = data[data["species"].ne("")].copy()
data = data[
    data["decimalLatitude"].between(-90, 90)
    & data["decimalLongitude"].between(-180, 180)
    & data["year"].between(TAHUN_AWAL, TAHUN_AKHIR)
].copy()
data["year"] = data["year"].astype(int)
log_cleaning.append(["Spesies/koordinat/tahun tidak valid atau di luar periode", sebelum - len(data), len(data)])

# 8) Kategori taksonomi yang kosong dipertahankan sebagai Unknown.
for col in ["kingdom", "class", "family"]:
    if col in data.columns:
        data[col] = data[col].fillna("Unknown").replace("", "Unknown")
    else:
        data[col] = "Unknown"

log_cleaning_df = pd.DataFrame(
    log_cleaning,
    columns=["Tahap penyaringan", "Jumlah baris dikeluarkan", "Sisa baris"]
)
display(log_cleaning_df)

print("Jumlah data awal:", f"{jumlah_awal:,}")
print("Jumlah data setelah cleaning:", f"{len(data):,}")
print("Missing values pada kolom inti:")
display(data[["species", "kingdom", "family", "class", "decimalLatitude", "decimalLongitude", "year"]].isna().sum().to_frame("Jumlah kosong"))

,Tahap penyaringan,Jumlah baris dikeluarkan,Sisa baris
0,Duplikasi gbifID,0,2022875
1,Bukan countryCode ID,0,2022875
2,Bukan taxonRank SPECIES,333318,1689557
3,Spesies/koordinat/tahun tidak valid atau di lu...,0,1689557


Jumlah data awal: 2,022,875
Jumlah data setelah cleaning: 1,689,557
Missing values pada kolom inti:


,Jumlah kosong
species,0
kingdom,0
family,0
class,0
decimalLatitude,0
decimalLongitude,0
year,0


## 4. Pemeriksaan Cakupan Wilayah dan Keterbatasan Geografis

Penyaringan `countryCode = ID` dan koordinat valid memastikan catatan berada dalam cakupan geografis Indonesia menurut interpretasi GBIF. Namun, filter negara tidak otomatis memisahkan daratan dari laut; catatan pesisir atau perairan Indonesia dapat tetap ada dan harus dicatat sebagai keterbatasan cakupan.

In [ ]:
print("Rentang latitude:", data["decimalLatitude"].min(), "s.d.", data["decimalLatitude"].max())
print("Rentang longitude:", data["decimalLongitude"].min(), "s.d.", data["decimalLongitude"].max())
print("Rentang tahun:", data["year"].min(), "s.d.", data["year"].max())

if "depth" in data.columns:
    print("Catatan dengan nilai depth terisi:", f"{data['depth'].notna().sum():,}")
    print("Catatan tanpa nilai depth:", f"{data['depth'].isna().sum():,}")

print("Catatan metodologis: batas negara GBIF bukan poligon daratan. Untuk pemetaan daratan yang ketat, lakukan overlay dengan batas pulau/daratan Indonesia pada tahap lanjutan.")

Rentang latitude: -49.345 s.d. 19.43845
Rentang longitude: -155.88567 s.d. 144.9
Rentang tahun: 2015 s.d. 2025
Catatan dengan nilai depth terisi: 5,824
Catatan tanpa nilai depth: 1,683,733
Catatan metodologis: batas negara GBIF bukan poligon daratan. Untuk pemetaan daratan yang ketat, lakukan overlay dengan batas pulau/daratan Indonesia pada tahap lanjutan.


## 5. Feature Engineering – Membentuk Grid dan Tahun Pengamatan

Koordinat diubah menjadi indeks grid 0,5° × 0,5°. Data kemudian diagregasi berdasarkan `grid_id` dan `year` sehingga satu baris mewakili satu unit wilayah pada satu tahun pengamatan.

In [ ]:
# Membuat indeks grid geografis.
data["grid_lat_bin"] = np.floor(data["decimalLatitude"] / GRID_SIZE).astype(int)
data["grid_lon_bin"] = np.floor(data["decimalLongitude"] / GRID_SIZE).astype(int)

data["grid_lat_center"] = (data["grid_lat_bin"] + 0.5) * GRID_SIZE
data["grid_lon_center"] = (data["grid_lon_bin"] + 0.5) * GRID_SIZE
data["grid_id"] = (
    data["grid_lat_bin"].astype(str) + "_" + data["grid_lon_bin"].astype(str)
)

# Menyederhanakan kingdom menjadi kategori komposisi yang stabil.
data["kingdom_group"] = data["kingdom"].astype("string").str.strip().str.title()
data["kingdom_group"] = data["kingdom_group"].fillna("Unknown")
data.loc[data["kingdom_group"].eq(""), "kingdom_group"] = "Unknown"
data.loc[
    ~data["kingdom_group"].isin(["Animalia", "Plantae", "Fungi", "Unknown"]),
    "kingdom_group"
] = "Other"

print("Contoh penempatan catatan ke grid:")
display(data[["decimalLatitude", "decimalLongitude", "grid_id", "grid_lat_center", "grid_lon_center", "year", "kingdom_group"]].head())

Contoh penempatan catatan ke grid:


,decimalLatitude,decimalLongitude,grid_id,grid_lat_center,grid_lon_center,year,kingdom_group
0,-1.770700,104.201890,-4_208,-1.75,104.25,2020,Plantae
1,0.473349,121.437461,0_242,0.25,121.25,2023,Animalia
2,-0.414129,121.865931,-1_243,-0.25,121.75,2023,Animalia
3,0.914723,127.547892,1_255,0.75,127.75,2023,Animalia
4,-7.158979,107.430856,-15_214,-7.25,107.25,2023,Animalia


## 6. Feature Engineering – Agregasi Karakteristik Biodiversitas

Pada tahap ini dihitung jumlah catatan, jumlah spesies unik, jumlah famili unik, jumlah kelas unik, dan proporsi catatan menurut kelompok kingdom. Fitur tersebut menggambarkan karakteristik dokumentasi biodiversitas untuk setiap grid dan tahun.

In [ ]:
group_keys = ["grid_id", "grid_lat_center", "grid_lon_center", "year"]

# Agregasi jumlah catatan dan kekayaan taksonomi.
ringkasan_grid = data.groupby(group_keys, dropna=False).agg(
    total_records=("gbifID", "nunique"),
    species_richness=("species", "nunique"),
    family_richness=("family", lambda s: s[s.ne("Unknown")].nunique()),
    class_richness=("class", lambda s: s[s.ne("Unknown")].nunique())
).reset_index()

# Menghitung proporsi setiap kelompok kingdom pada grid-tahun.
komposisi = pd.crosstab(
    index=[data["grid_id"], data["year"]],
    columns=data["kingdom_group"],
    normalize="index"
)
komposisi = komposisi.rename(columns={
    "Animalia": "prop_animalia",
    "Plantae": "prop_plantae",
    "Fungi": "prop_fungi",
    "Other": "prop_other",
    "Unknown": "prop_unknown"
})

kolom_proporsi = ["prop_animalia", "prop_plantae", "prop_fungi", "prop_other", "prop_unknown"]
for col in kolom_proporsi:
    if col not in komposisi.columns:
        komposisi[col] = 0.0
komposisi = komposisi[kolom_proporsi].reset_index()

# Gabungkan agregasi dan komposisi kingdom.
grid_year = ringkasan_grid.merge(
    komposisi,
    on=["grid_id", "year"],
    how="left"
)

print("Ukuran dataset agregasi (baris, kolom):", grid_year.shape)
print("Jumlah grid unik:", grid_year["grid_id"].nunique())
print("Jumlah kombinasi grid-tahun:", len(grid_year))
display(grid_year.head())

Ukuran dataset agregasi (baris, kolom): (7465, 13)
Jumlah grid unik: 1611
Jumlah kombinasi grid-tahun: 7465


,grid_id,grid_lat_center,grid_lon_center,year,total_records,species_richness,family_richness,class_richness,prop_animalia,prop_plantae,prop_fungi,prop_other,prop_unknown
0,-10_201,-4.75,100.75,2016,6,1,1,1,1.000000,0.000000,0.0,0.0,0.0
1,-10_204,-4.75,102.25,2019,1,1,1,1,1.000000,0.000000,0.0,0.0,0.0
2,-10_204,-4.75,102.25,2022,3,2,1,1,1.000000,0.000000,0.0,0.0,0.0
3,-10_205,-4.75,102.75,2024,1,1,1,1,1.000000,0.000000,0.0,0.0,0.0
4,-10_206,-4.75,103.25,2017,13,13,11,3,0.923077,0.076923,0.0,0.0,0.0


## 7. Menentukan Fitur Model dan Metadata

Model clustering menggunakan metrik jumlah catatan dan kekayaan taksonomi serta proporsi kelompok kingdom. Koordinat grid dan tahun disimpan sebagai metadata untuk pemetaan dan analisis tren, bukan sebagai fitur model agar hasil cluster tidak sekadar didominasi kedekatan geografis.

In [ ]:
# Log transform mengurangi kemencengan distribusi fitur hitungan.
for col in ["total_records", "species_richness", "family_richness", "class_richness"]:
    grid_year["log_" + col] = np.log1p(grid_year[col])

feature_cols = [
    "log_total_records",
    "log_species_richness",
    "log_family_richness",
    "log_class_richness",
    "prop_animalia",
    "prop_plantae",
    "prop_fungi",
    "prop_other",
    "prop_unknown"
]
metadata_cols = [
    "grid_id", "grid_lat_center", "grid_lon_center", "year",
    "total_records", "species_richness", "family_richness", "class_richness"
]

X = grid_year[feature_cols].replace([np.inf, -np.inf], np.nan)
metadata = grid_year[metadata_cols].copy()

print("Fitur yang digunakan untuk clustering:")
for f in feature_cols:
    print("-", f)
print("\nMetadata yang tidak menjadi input model:", [c for c in metadata_cols if c not in feature_cols])
display(X.describe().T)

Fitur yang digunakan untuk clustering:
- log_total_records
- log_species_richness
- log_family_richness
- log_class_richness
- prop_animalia
- prop_plantae
- prop_fungi
- prop_other
- prop_unknown

Metadata yang tidak menjadi input model: ['grid_id', 'grid_lat_center', 'grid_lon_center', 'year', 'total_records', 'species_richness', 'family_richness', 'class_richness']


,count,mean,std,min,25%,50%,75%,max
log_total_records,7465.0,3.013390,2.018601,0.693147,1.098612,2.639057,4.382027,10.310551
log_species_richness,7465.0,2.634899,1.604391,0.693147,1.098612,2.484907,3.931826,7.338238
log_family_richness,7465.0,2.337801,1.343080,0.693147,1.098612,2.197225,3.433987,6.188264
log_class_richness,7465.0,1.349954,0.688679,0.000000,0.693147,1.098612,1.791759,3.737670
prop_animalia,7465.0,0.869741,0.275902,0.000000,0.916667,1.000000,1.000000,1.000000
prop_plantae,7465.0,0.123246,0.268320,0.000000,0.000000,0.000000,0.072464,1.000000
prop_fungi,7465.0,0.003331,0.046318,0.000000,0.000000,0.000000,0.000000,1.000000
prop_other,7465.0,0.003672,0.053272,0.000000,0.000000,0.000000,0.000000,1.000000
prop_unknown,7465.0,0.000011,0.000441,0.000000,0.000000,0.000000,0.000000,0.022727


## 8. Data Splitting – Training, Validation, dan Test Berdasarkan Blok Spasial

Karena project menggunakan clustering tanpa label kelas, pembagian data tidak menggunakan stratifikasi. Grid dikelompokkan ke dalam blok geografis 2° agar grid yang berdekatan tidak tersebar ke bagian training, validation, dan test secara acak.

In [ ]:
# Membentuk blok geografis 2 derajat untuk menjaga pemisahan kelompok lokasi.
grid_year["spatial_block"] = (
    np.floor(grid_year["grid_lat_center"] / 2).astype(int).astype(str)
    + "_"
    + np.floor(grid_year["grid_lon_center"] / 2).astype(int).astype(str)
)
groups = grid_year["spatial_block"]

if groups.nunique() < 5:
    raise ValueError("Jumlah blok spasial terlalu sedikit untuk pembagian train/validation/test. Tinjau ukuran blok atau cakupan data.")

# Split pertama: 70% training, 30% data sementara.
split1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE)
train_idx, temp_idx = next(split1.split(X, groups=groups))

# Split kedua: data sementara dibagi dua menjadi validation dan test.
X_temp = X.iloc[temp_idx]
groups_temp = groups.iloc[temp_idx]
split2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=RANDOM_STATE)
val_rel_idx, test_rel_idx = next(split2.split(X_temp, groups=groups_temp))
val_idx = temp_idx[val_rel_idx]
test_idx = temp_idx[test_rel_idx]

X_train = X.iloc[train_idx].copy()
X_val = X.iloc[val_idx].copy()
X_test = X.iloc[test_idx].copy()
meta_train = metadata.iloc[train_idx].copy()
meta_val = metadata.iloc[val_idx].copy()
meta_test = metadata.iloc[test_idx].copy()

ringkasan_split = pd.DataFrame({
    "Subset": ["Training", "Validation", "Test"],
    "Jumlah sampel": [len(train_idx), len(val_idx), len(test_idx)],
    "Persentase": [len(train_idx)/len(X)*100, len(val_idx)/len(X)*100, len(test_idx)/len(X)*100],
    "Jumlah blok spasial": [groups.iloc[train_idx].nunique(), groups.iloc[val_idx].nunique(), groups.iloc[test_idx].nunique()]
})
display(ringkasan_split.round(2))

# Pastikan blok spasial tidak tumpang tindih antar subset.
blok_train = set(groups.iloc[train_idx])
blok_val = set(groups.iloc[val_idx])
blok_test = set(groups.iloc[test_idx])
assert blok_train.isdisjoint(blok_val)
assert blok_train.isdisjoint(blok_test)
assert blok_val.isdisjoint(blok_test)
print("Validasi berhasil: blok spasial train, validation, dan test tidak tumpang tindih.")

,Subset,Jumlah sampel,Persentase,Jumlah blok spasial
0,Training,4744,63.55,135
1,Validation,1505,20.16,29
2,Test,1216,16.29,29


Validasi berhasil: blok spasial train, validation, dan test tidak tumpang tindih.


## 9. Data Transformation – Imputasi dan Standardisasi

Setelah data dibagi, nilai kosong pada fitur numerik diimputasi menggunakan median dan seluruh fitur distandarisasi dengan StandardScaler. Parameter imputasi dan scaler hanya dipelajari dari training, kemudian diterapkan ke validation dan test untuk mencegah data leakage.

In [ ]:
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

# Fit hanya pada data training.
X_train_imputed = imputer.fit_transform(X_train)
X_train_scaled = scaler.fit_transform(X_train_imputed)

# Validation dan test hanya di-transform.
X_val_imputed = imputer.transform(X_val)
X_val_scaled = scaler.transform(X_val_imputed)
X_test_imputed = imputer.transform(X_test)
X_test_scaled = scaler.transform(X_test_imputed)

print("Shape fitur training:", X_train_scaled.shape)
print("Shape fitur validation:", X_val_scaled.shape)
print("Shape fitur test:", X_test_scaled.shape)
print("Jumlah fitur model:", len(feature_cols))
print("Imputer dan scaler di-fit hanya pada data training.")

Shape fitur training: (4744, 9)
Shape fitur validation: (1505, 9)
Shape fitur test: (1216, 9)
Jumlah fitur model: 9
Imputer dan scaler di-fit hanya pada data training.


## 10. Data Balancing – Tidak Diterapkan pada Clustering

Data balancing seperti SMOTE atau undersampling kelas tidak dilakukan karena dataset tidak memiliki label kelas target. Clustering bertujuan menemukan kelompok secara otomatis; mengubah proporsi kelas seperti pada klasifikasi tidak sesuai dengan rancangan ini.

In [ ]:
print("Data balancing tidak diterapkan.")
print("Alasan: project menggunakan unsupervised learning tanpa label target.")
print("Perbedaan jumlah catatan antarwilayah dipertahankan sebagai karakteristik data, lalu fitur jumlah ditransformasi log1p dan distandarisasi.")

Data balancing tidak diterapkan.
Alasan: project menggunakan unsupervised learning tanpa label target.
Perbedaan jumlah catatan antarwilayah dipertahankan sebagai karakteristik data, lalu fitur jumlah ditransformasi log1p dan distandarisasi.


## 11. Menyimpan Hasil Data Preparation

Bagian ini menyimpan fitur dan metadata train/validation/test, matriks fitur yang telah distandarisasi, daftar fitur, serta objek imputer dan scaler. Berkas tersebut dapat digunakan kembali pada notebook pemodelan berikutnya.

In [ ]:
# Simpan metadata dan fitur asli per subset.
train_export = meta_train.reset_index(drop=True).join(X_train.reset_index(drop=True))
val_export = meta_val.reset_index(drop=True).join(X_val.reset_index(drop=True))
test_export = meta_test.reset_index(drop=True).join(X_test.reset_index(drop=True))

train_export.to_csv(OUTPUT_DIR / "train_grid_year.csv", index=False)
val_export.to_csv(OUTPUT_DIR / "validation_grid_year.csv", index=False)
test_export.to_csv(OUTPUT_DIR / "test_grid_year.csv", index=False)

# Simpan matriks fitur terstandarisasi.
np.save(OUTPUT_DIR / "X_train_scaled.npy", X_train_scaled)
np.save(OUTPUT_DIR / "X_validation_scaled.npy", X_val_scaled)
np.save(OUTPUT_DIR / "X_test_scaled.npy", X_test_scaled)

# Simpan metadata, fitur, dan preprocessing untuk penggunaan ulang.
joblib.dump(imputer, OUTPUT_DIR / "imputer_train_only.joblib")
joblib.dump(scaler, OUTPUT_DIR / "scaler_train_only.joblib")

with open(OUTPUT_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, ensure_ascii=False, indent=2)

ringkasan_split.to_csv(OUTPUT_DIR / "split_summary.csv", index=False)
log_cleaning_df.to_csv(OUTPUT_DIR / "cleaning_log.csv", index=False)

print("Seluruh hasil preparation tersimpan pada:", OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    print("-", p.name, f"({p.stat().st_size / (1024**2):.2f} MB)")

Seluruh hasil preparation tersimpan pada: /content/hasil_data_preparation_clustering
- X_test_scaled.npy (0.08 MB)
- X_train_scaled.npy (0.33 MB)
- X_validation_scaled.npy (0.10 MB)
- cleaning_log.csv (0.00 MB)
- feature_columns.json (0.00 MB)
- imputer_train_only.joblib (0.00 MB)
- scaler_train_only.joblib (0.00 MB)
- split_summary.csv (0.00 MB)
- test_grid_year.csv (0.16 MB)
- train_grid_year.csv (0.63 MB)
- validation_grid_year.csv (0.21 MB)


## 12. Ringkasan Data Preparation (Apa–Mengapa–Bagaimana)

Bagian ini merangkum proses perubahan data mentah GBIF menjadi dataset grid-tahun yang siap digunakan oleh model clustering.

In [ ]:
ringkasan_prep = pd.DataFrame([
    ["Apa", "Membersihkan catatan GBIF, membentuk grid geografis 0,5 derajat, mengagregasi karakteristik spesies, membagi data secara spasial, lalu melakukan imputasi dan standardisasi."],
    ["Mengapa", "Agar setiap sampel mewakili satu unit wilayah-tahun dengan fitur numerik yang sesuai untuk K-Means, GMM, dan DEC serta mengurangi kebocoran spasial dan data leakage preprocessing."],
    ["Bagaimana", "Menggunakan Pandas/NumPy untuk cleaning dan agregasi, GroupShuffleSplit untuk pembagian blok spasial, serta SimpleImputer dan StandardScaler yang di-fit hanya pada data training."],
], columns=["Komponen", "Ringkasan"])
display(ringkasan_prep)

print("\nRingkasan akhir:")
print("Data mentah:", f"{jumlah_awal:,}")
print("Data catatan setelah cleaning:", f"{len(data):,}")
print("Unit analisis grid-tahun:", f"{len(grid_year):,}")
print("Fitur clustering:", len(feature_cols))
print("Data balancing: tidak diterapkan")
print("Notebook 02 selesai. Dataset siap digunakan pada tahap Model Design.")

,Komponen,Ringkasan
0,Apa,"Membersihkan catatan GBIF, membentuk grid geog..."
1,Mengapa,Agar setiap sampel mewakili satu unit wilayah-...
2,Bagaimana,Menggunakan Pandas/NumPy untuk cleaning dan ag...



Ringkasan akhir:
Data mentah: 2,022,875
Data catatan setelah cleaning: 1,689,557
Unit analisis grid-tahun: 7,465
Fitur clustering: 9
Data balancing: tidak diterapkan
Notebook 02 selesai. Dataset siap digunakan pada tahap Model Design.
